In [6]:
import pandas as pd
import numpy as np

# ==== Cấu hình ====
file_path = "/dis/DS/hungnt/FLBased-ICS-NIDS/Labeled_CICMODBUS2023.csv"
output_path = "/dis/DS/hungnt/CICModbus2023/Labeled_CICMODBUS2023_cleaned.csv"

# ==== Đọc file ====
df = pd.read_csv(file_path)

# ==== Chuẩn hóa tên cột ====
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# ==== Xóa các cột toàn NaN ====
df = df.dropna(axis=1, how="all")

# ==== Loại bỏ các cột cố định (chỉ có 1 giá trị duy nhất) ====
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
if constant_cols:
    print(f"Loại bỏ {len(constant_cols)} cột cố định: {constant_cols}")
    df = df.drop(columns=constant_cols)

# ==== Giữ lại các cột số (trừ Label) ====
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if "Label" in df.columns:
    final_cols = numeric_cols + ["Label"]
else:
    raise ValueError("Không tìm thấy cột 'Label' trong file!")

df = df[final_cols]

# ==== Mã hóa cột Label ====
df["Label"] = df["Label"].astype(str).str.strip().str.upper()

unique_Labels = sorted(df["Label"].unique())

if "BENIGN" in unique_Labels:
    unique_Labels.remove("BENIGN")
    Label_order = ["BENIGN"] + unique_Labels
else:
    Label_order = unique_Labels

Label2num = {lbl: i for i, lbl in enumerate(Label_order)}
df["Label"] = df["Label"].map(Label2num)

# ==== Xuất file kết quả ====
cols_to_drop_manually = ['Src_Port', 'Dst_Port']
df_new = df.drop(cols_to_drop_manually, axis=1)
df_new.to_csv(output_path, index=False)

# ==== Báo cáo ====
print("\n Hoàn tất xử lý!")
print(f" Số cột còn lại: {len(df_new.columns)}")
print(f" File đã lưu: {output_path}")
print(f" Mapping nhãn ({len(Label2num)} loại):")
for k, v in Label2num.items():
    print(f"  {k} → {v}")

# ==== Thống kê phân bố nhãn ====
print("\n Phân bố nhãn sau mã hóa:")
print(df_new["Label"].value_counts().sort_index())

/tmp/ipykernel_180277/1406031996.py:9: DtypeWarning: Columns (94) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Loại bỏ 7 cột cố định: ['Fwd_URG_Flags', 'Bwd_URG_Flags', 'URG_Flag_Count', 'CWR_Flag_Count', 'ECE_Flag_Count', 'ICMP_Code', 'ICMP_Type']

 Hoàn tất xử lý!
 Số cột còn lại: 81
 File đã lưu: /dis/DS/hungnt/CICModbus2023/Labeled_CICMODBUS2023_cleaned.csv
 Mapping nhãn (9 loại):
  BENIGN → 0
  BASELINE REPLAY → 1
  BRUTE FORCE → 2
  DELAY RESPONSE → 3
  FRAME STACKING → 4
  LENGTH MANIPULATION → 5
  PAYLOAD INJECTION → 6
  QUERY FLOODING → 7
  RECON → 8

 Phân bố nhãn sau mã hóa:
Label
0    8903738
1       1836
2    1318212
3         86
4       1520
5       1404
6       1490
7       1511
8       1305
Name: count, dtype: int64


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_N_PER_LABEL = 600
DATA_PATH = '/dis/DS/hungnt/CICModbus2023/Labeled_CICMODBUS2023_cleaned.csv'
TRAIN_OUT = '/dis/DS/hungnt/CICModbus2023/train_70_600.csv'
TEST_OUT  = '/dis/DS/hungnt/CICModbus2023/test_30_600.csv'

# 1) Đọc dữ liệu
df = pd.read_csv(DATA_PATH)

# 2) Lấy đúng 6000 mẫu label 0 và 6000 mẫu label 2 cho TEST
counts = df['Label'].value_counts()
for lab in [0, 2]:
    if counts.get(lab, 0) < TEST_N_PER_LABEL:
        raise ValueError(f"Số mẫu của nhãn {lab} < {TEST_N_PER_LABEL}. Hiện có {counts.get(lab, 0)}.")

test_label_0 = df[df['Label'] == 0].sample(n=TEST_N_PER_LABEL, random_state=RANDOM_STATE)
test_label_2 = df[df['Label'] == 2].sample(n=TEST_N_PER_LABEL, random_state=RANDOM_STATE)

# 3) Loại các mẫu đã đưa vào test khỏi dữ liệu gốc
remaining = df.drop(test_label_0.index).drop(test_label_2.index)

# 4) Phần còn lại của nhãn 0 & 2 => ĐƯA HẾT VÀO TRAIN (không chia)
train_from_0_2 = remaining[remaining['Label'].isin([0, 2])]

# 5) Các nhãn khác (không phải 0,2) => CHIA 70/30 có stratify
others = remaining[~remaining['Label'].isin([0, 2])]
train_others, test_others = train_test_split(
    others,
    test_size=0.3,
    stratify=others['Label'],
    random_state=RANDOM_STATE
)

# 6) Gộp lại train/test cuối
train_final = pd.concat([train_others, train_from_0_2], ignore_index=True)
test_final  = pd.concat([test_others, test_label_0, test_label_2], ignore_index=True)

# 7) Lưu file
train_final.to_csv(TRAIN_OUT, index=False)
test_final.to_csv(TEST_OUT, index=False)

# 8) Kiểm tra
print("Tổng train:", len(train_final))
print("Tổng test:", len(test_final))

print("\nPhân bố nhãn trong train:")
print(train_final['Label'].value_counts().sort_index())

print("\nPhân bố nhãn trong test:")
print(test_final['Label'].value_counts().sort_index())


Tổng train: 10227156
Tổng test: 3946

Phân bố nhãn trong train:
Label
0    8903138
1       1285
2    1317612
3         60
4       1064
5        983
6       1043
7       1058
8        913
Name: count, dtype: int64

Phân bố nhãn trong test:
Label
0    600
1    551
2    600
3     26
4    456
5    421
6    447
7    453
8    392
Name: count, dtype: int64
